CBDC Classifier - bert-base-uncased model


In [1]:
# 1. Install and upgrade dependencies
# !pip install -U transformers datasets scikit-learn tqdm accelerate --quiet

# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 3. Import libraries
import pandas as pd
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import set_seed
set_seed(42)

# 4. Load data DIRECTLY from GitHub (RAW link)
github_csv_url = "https://raw.githubusercontent.com/bilalezafar/CentralBank-BERT/main/2-CBDC-BERT/bert_training_data.csv"
df = pd.read_csv(github_csv_url)

# Keep only usable columns and clean
df = df[["sentence", "label"]].dropna()
df["sentence"] = df["sentence"].astype(str)
df["label"] = df["label"].astype(int)

print(f"Loaded {len(df)} rows: {df.columns.tolist()}")
print(df.head())
print("Label counts:\n", df["label"].value_counts())

# 5. Train/Test split
train_df, test_df = train_test_split(
    df[["sentence", "label"]],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))

# 6. Tokenization (BERT base uncased)
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(batch):
    return tokenizer(batch["sentence"], padding="max_length", truncation=True, max_length=128)

print("Tokenizing train...")
train_ds = train_ds.map(tokenize, batched=True, batch_size=256, desc="Tokenizing train")
print("Tokenizing test...")
test_ds  = test_ds.map(tokenize,  batched=True, batch_size=256, desc="Tokenizing test")

train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# 7. Load BERT for classification
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2
)

# 8. Training arguments
training_args = TrainingArguments(
    output_dir="/content/cbdc-bert-uncased",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_dir="/content/logs",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="steps",
    save_total_limit=2,
    report_to="none"
)

# 9. Compute metrics function
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

# 10. Trainer & training
print("Training starting...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
print("Training complete.")

# 11. Evaluate & Save to Google Drive: My Drive > CBDC Project > cbdc-bert-uncased
print("Final evaluation on test set:")
eval_result = trainer.evaluate()
print(eval_result)

save_path = "/content/drive/MyDrive/CBDC Project/cbdc-bert-uncased"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved to: {save_path}")


Mounted at /content/drive
Loaded 11000 rows: ['sentence', 'label']
                                            sentence  label
0  central banks are now carefully monitoring new...      1
1  (2020), bundesbank round-up, deutsche bundesba...      0
2  these three major improvements - namely foster...      0
3  this central bank digital currency is being us...      1
4  almost all of that attention is focused on ret...      1
Label counts:
 label
0    5610
1    5390
Name: count, dtype: int64
Train: 8800, Test: 2200


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Training starting...


/tmp/ipython-input-3847055247.py:94: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.151100,0.159671,0.958636,0.959321
2,0.002400,0.042242,0.992273,0.992133
3,0.000100,0.037562,0.993182,0.993059


Training complete.
Final evaluation on test set:


{'eval_loss': 0.03756208345293999, 'eval_accuracy': 0.9931818181818182, 'eval_f1': 0.993058769088385, 'eval_runtime': 15.1279, 'eval_samples_per_second': 145.426, 'eval_steps_per_second': 18.178, 'epoch': 3.0}
Model saved to: /content/drive/MyDrive/CBDC Project/cbdc-bert-uncased


CBDC Stance - bert-base-uncased model

In [2]:
# =========================
# 0) Setup
# =========================
# !pip -q install -U transformers datasets accelerate scikit-learn

import os, json, numpy as np, pandas as pd, torch, torch.nn as nn
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, WeightedRandomSampler
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer, EarlyStoppingCallback
)

# -------------------------
# 1) Paths (Drive save locations)
# -------------------------
MOUNT              = "/content/drive"
DRIVE_PROJECT_DIR  = f"{MOUNT}/MyDrive/CBDC Project"   # <- My Drive > CBDC Project
OUTPUT_DIR         = os.path.join(DRIVE_PROJECT_DIR, "checkpoints_cbdc_stance_bert_uncased")
FINAL_MODEL_DIR    = os.path.join(DRIVE_PROJECT_DIR, "cbdc-stance-bert-uncased")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

# -------------------------
# 2) Mount Drive & load data from GitHub (RAW)
# -------------------------
try:
    drive.mount(MOUNT)
except Exception:
    pass

DATA_RAW_URL = "https://raw.githubusercontent.com/bilalezafar/CentralBank-BERT/main/3-CBDC-Stance/stance_sentences.csv"
df = pd.read_csv(DATA_RAW_URL)  # expects columns: text, label

df = df[["text", "label"]].dropna().reset_index(drop=True)
df["text"] = df["text"].astype(str)

# -------------------------
# 3) Three-class labels
# -------------------------
label_list = ["Anti-CBDC", "Pro-CBDC", "Wait-and-See"]
df = df[df["label"].isin(label_list)].copy()

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
df["label_id"] = df["label"].map(label2id)

print("Label counts (3-class):\n", df["label"].value_counts())

# -------------------------
# 4) Train/Val/Test split (stratified)
# -------------------------
train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df["label_id"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label_id"], random_state=42
)

def to_hf(ds):
    return Dataset.from_pandas(
        ds[["text","label_id"]].rename(columns={"label_id":"labels"}),
        preserve_index=False
    )

dataset = DatasetDict({
    "train": to_hf(train_df),
    "validation": to_hf(val_df),
    "test": to_hf(test_df),
})

# -------------------------
# 5) Tokenizer & collator (BERT base uncased)
# -------------------------
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=320)

dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"],
    load_from_cache_file=False
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# -------------------------
# 6) Model (BERT base uncased)
# -------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# -------------------------
# 7) Imbalance handling: sampler (gentle)
# -------------------------
class_counts = train_df["label_id"].value_counts().to_dict()
inv_class_counts = {c: 1.0 / count for c, count in class_counts.items()}
sample_weights = train_df["label_id"].map(inv_class_counts).astype("float32").values
sample_weights = np.sqrt(sample_weights).astype("float32")
sample_weights_t = torch.tensor(sample_weights)

# -------------------------
# 8) Focal Loss (soft)
# -------------------------
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 1.0, weight=None, reduction: str = "mean"):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
        probs = log_probs.exp()
        ce = torch.nn.functional.nll_loss(
            log_probs, targets.long(), weight=self.weight, reduction="none"
        )
        pt = probs.gather(dim=-1, index=targets.view(-1,1)).squeeze(1)
        loss = (1 - pt).pow(self.gamma) * ce
        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss

# -------------------------
# 9) Custom Trainer: Focal + WeightedRandomSampler
# -------------------------
class FocalSamplerTrainer(Trainer):
    def __init__(self, class_weights=None, focal_gamma: float = 1.0, train_sample_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        device = self.args.device
        self.class_weights = class_weights.to(device) if class_weights is not None else None
        self.criterion = FocalLoss(gamma=focal_gamma, weight=self.class_weights)
        self.train_sample_weights = train_sample_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = self.criterion(logits, labels)
        return (loss, outputs) if return_outputs else loss

    def get_train_dataloader(self) -> DataLoader:
        if self.train_dataset is None:
            raise ValueError("Trainer: training requires a train_dataset.")
        if self.train_sample_weights is None:
            return super().get_train_dataloader()
        sampler = WeightedRandomSampler(
            weights=self.train_sample_weights,
            num_samples=len(self.train_sample_weights),
            replacement=True
        )
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            drop_last=self.args.dataloader_drop_last,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
            persistent_workers=self.args.dataloader_persistent_workers,
        )

# -------------------------
# 10) Metrics
# -------------------------
def compute_metrics(eval_pred):
    labels = eval_pred.label_ids
    preds = eval_pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# -------------------------
# 11) Training args
# -------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    num_train_epochs=8,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    warmup_ratio=0.06,
    fp16=True,
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
    seed=42,
    dataloader_num_workers=0,
)

trainer = FocalSamplerTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=None,                 # sampler handles imbalance
    focal_gamma=1.0,
    train_sample_weights=sample_weights_t,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# -------------------------
# 12) Train
# -------------------------
print("🚀 Training…")
trainer.train()
print("✅ Training complete.")

# -------------------------
# 13) Evaluate on val & test
# -------------------------
print("📊 Validation:", trainer.evaluate(dataset["validation"]))
test_metrics = trainer.evaluate(dataset["test"])
print("📊 Test:", test_metrics)

preds = trainer.predict(dataset["test"])
y_true = preds.label_ids
y_pred = preds.predictions.argmax(-1)
print("\nClassification report (test):\n",
      classification_report(y_true, y_pred, target_names=label_list, digits=4))

# -------------------------
# 14) Save model + tokenizer + label map
# -------------------------
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
with open(os.path.join(FINAL_MODEL_DIR, "label_mapping.json"), "w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, indent=2)

print(f"\n💾 Saved best model to: {FINAL_MODEL_DIR}")
print(f"   Checkpoints in: {OUTPUT_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Label counts (3-class):
 label
Pro-CBDC        742
Wait-and-See    694
Anti-CBDC       211
Name: count, dtype: int64


Map:   0%|          | 0/1317 [00:00<?, ? examples/s]

Map:   0%|          | 0/165 [00:00<?, ? examples/s]

Map:   0%|          | 0/165 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-860590269.py:143: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `FocalSamplerTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


🚀 Training…


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.679800,0.475447,0.648485,0.631012,0.654552
2,0.252500,0.458055,0.727273,0.698289,0.733154
3,0.190500,0.464142,0.733333,0.690685,0.734821
4,0.083800,0.489814,0.775758,0.738535,0.779216
5,0.070400,0.509161,0.763636,0.724754,0.766548
6,0.044000,0.548005,0.781818,0.745295,0.786586
7,0.021400,0.583884,0.757576,0.727407,0.760404
8,0.011300,0.584479,0.757576,0.730160,0.759566


✅ Training complete.


📊 Validation: {'eval_loss': 0.5480045080184937, 'eval_accuracy': 0.7818181818181819, 'eval_f1_macro': 0.7452954170511422, 'eval_f1_weighted': 0.7865864415482736, 'eval_runtime': 0.3062, 'eval_samples_per_second': 538.816, 'eval_steps_per_second': 35.921, 'epoch': 8.0}
📊 Test: {'eval_loss': 0.5277591943740845, 'eval_accuracy': 0.793939393939394, 'eval_f1_macro': 0.7841179813700704, 'eval_f1_weighted': 0.791153051616193, 'eval_runtime': 0.2753, 'eval_samples_per_second': 599.437, 'eval_steps_per_second': 39.962, 'epoch': 8.0}

Classification report (test):
               precision    recall  f1-score   support

   Anti-CBDC     0.6923    0.8571    0.7660        21
    Pro-CBDC     0.7791    0.8933    0.8323        75
Wait-and-See     0.8679    0.6667    0.7541        69

    accuracy                         0.7939       165
   macro avg     0.7798    0.8057    0.7841       165
weighted avg     0.8052    0.7939    0.7912       165


💾 Saved best model to: /content/drive/MyDrive/CBDC Proje

CBDC Sentiment - bert-base-uncased model

In [3]:
# =========================
# 0) Setup
# =========================
# !pip -q install -U transformers datasets accelerate scikit-learn

import os, json, numpy as np, pandas as pd, torch, torch.nn as nn
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, WeightedRandomSampler
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer, EarlyStoppingCallback
)

# -------------------------
# 1) Paths (Drive save locations)
# -------------------------
MOUNT             = "/content/drive"
BASE_DIR          = f"{MOUNT}/MyDrive/CBDC Project"   # My Drive > CBDC Project

OUTPUT_DIR        = os.path.join(BASE_DIR, "checkpoints_cbdc_sentiment_bert_uncased")
FINAL_MODEL_DIR   = os.path.join(BASE_DIR, "cbdc-sentiment-bert-uncased")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

# -------------------------
# 2) Mount Drive & load data from GitHub (RAW)
# -------------------------
try:
    drive.mount(MOUNT)
except Exception:
    pass

DATA_RAW_URL = "https://raw.githubusercontent.com/bilalezafar/CentralBank-BERT/main/4-CBDC-Sentiment/cbdc_sentiment_training.csv"
df = pd.read_csv(DATA_RAW_URL)  # expects columns: url, sentence, label

df = df[["sentence", "label"]].dropna().reset_index(drop=True)
df["sentence"] = df["sentence"].astype(str)
df["label"] = df["label"].astype(str).str.strip().str.lower()

# -------------------------
# 3) Three-class labels (standardized)
# -------------------------
label_list = ["negative", "neutral", "positive"]
df = df[df["label"].isin(label_list)].copy()

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
df["label_id"] = df["label"].map(label2id)

print("Label counts:\n", df["label"].value_counts())

# -------------------------
# 4) Train/Val/Test split (80/10/10 stratified)
# -------------------------
train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df["label_id"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label_id"], random_state=42
)

def to_hf(ds):
    return Dataset.from_pandas(
        ds[["sentence","label_id"]].rename(columns={"sentence":"text","label_id":"labels"}),
        preserve_index=False
    )

dataset = DatasetDict({
    "train": to_hf(train_df),
    "validation": to_hf(val_df),
    "test": to_hf(test_df),
})

# -------------------------
# 5) Tokenizer & collator (BERT base uncased)
# -------------------------
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=320)

dataset = dataset.map(
    tokenize, batched=True, remove_columns=["text"], load_from_cache_file=False
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# -------------------------
# 6) Model
# -------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# -------------------------
# 7) Imbalance handling
# -------------------------
labels_train = np.array(dataset["train"]["labels"])
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(label_list)),
    y=labels_train
)
class_weights = torch.tensor(weights, dtype=torch.float)
print("Class weights (used in loss):", {id2label[i]: float(w) for i, w in enumerate(class_weights)})

class_counts = np.bincount(labels_train, minlength=len(label_list))
inv_counts = {i: (1.0 / c) for i, c in enumerate(class_counts)}
instance_weights = np.array([inv_counts[l] for l in labels_train], dtype="float32")
instance_weights = np.sqrt(instance_weights).astype("float32")
train_sample_weights_t = torch.tensor(instance_weights)

# -------------------------
# 8) Focal Loss (soft: gamma=1.0)
# -------------------------
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 1.0, weight: torch.Tensor | None = None, reduction: str = "mean"):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
        probs = log_probs.exp()
        ce = torch.nn.functional.nll_loss(
            log_probs, targets.long(), weight=self.weight, reduction="none"
        )
        pt = probs.gather(dim=-1, index=targets.view(-1,1)).squeeze(1)
        loss = (1 - pt).pow(self.gamma) * ce
        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss

# -------------------------
# 9) Custom Trainer: Focal + WeightedRandomSampler
# -------------------------
class FocalSamplerTrainer(Trainer):
    def __init__(self, class_weights=None, focal_gamma: float = 1.0, train_sample_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        device = self.args.device
        self.class_weights = class_weights.to(device) if class_weights is not None else None
        self.criterion = FocalLoss(gamma=focal_gamma, weight=self.class_weights)
        self.train_sample_weights = train_sample_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = self.criterion(logits, labels)
        return (loss, outputs) if return_outputs else loss

    def get_train_dataloader(self) -> DataLoader:
        if self.train_dataset is None:
            raise ValueError("Trainer: training requires a train_dataset.")
        if self.train_sample_weights is None:
            return super().get_train_dataloader()
        sampler = WeightedRandomSampler(
            weights=self.train_sample_weights,
            num_samples=len(self.train_sample_weights),
            replacement=True
        )
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            drop_last=self.args.dataloader_drop_last,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
            persistent_workers=self.args.dataloader_persistent_workers,
        )

# -------------------------
# 10) Metrics
# -------------------------
def compute_metrics(eval_pred):
    labels = eval_pred.label_ids
    preds = eval_pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# -------------------------
# 11) Training args
# -------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    num_train_epochs=8,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    warmup_ratio=0.06,
    fp16=True,
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
    seed=42,
    dataloader_num_workers=0,
)

trainer = FocalSamplerTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,  # (use tokenizer here; "processing_class" can error depending on transformers version)
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    focal_gamma=1.0,
    train_sample_weights=train_sample_weights_t,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# -------------------------
# 12) Train
# -------------------------
print("🚀 Training…")
trainer.train()
print("✅ Training complete.")

# -------------------------
# 13) Evaluate on val & test
# -------------------------
print("📊 Validation:", trainer.evaluate(dataset["validation"]))
test_metrics = trainer.evaluate(dataset["test"])
print("📊 Test:", test_metrics)

preds = trainer.predict(dataset["test"])
y_true = preds.label_ids
y_pred = preds.predictions.argmax(-1)
print("\nClassification report (test):\n",
      classification_report(y_true, y_pred, target_names=label_list, digits=4))

# -------------------------
# 14) Save model + tokenizer + label map
# -------------------------
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
with open(os.path.join(FINAL_MODEL_DIR, "label_mapping.json"), "w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, indent=2)

print(f"\n💾 Saved best model to: {FINAL_MODEL_DIR}")
print(f"   Checkpoints in: {OUTPUT_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Label counts:
 label
neutral     1068
positive    1026
negative     311
Name: count, dtype: int64


Map:   0%|          | 0/1924 [00:00<?, ? examples/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/241 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3356239403.py:150: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `FocalSamplerTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Class weights (used in loss): {'negative': 2.5756359100341797, 'neutral': 0.7509757876396179, 'positive': 0.7811611890792847}
🚀 Training…


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.412000,0.386151,0.737500,0.732907,0.738401
2,0.285400,0.342908,0.745833,0.744674,0.741259
3,0.161000,0.383084,0.800000,0.786279,0.800035
4,0.072100,0.336486,0.804167,0.805801,0.804046
5,0.050400,0.401096,0.825000,0.813019,0.824858
6,0.046600,0.362010,0.820833,0.814692,0.820851
7,0.022800,0.437756,0.825000,0.828117,0.824767
8,0.023600,0.420651,0.820833,0.818776,0.820573


✅ Training complete.


📊 Validation: {'eval_loss': 0.4377555847167969, 'eval_accuracy': 0.825, 'eval_f1_macro': 0.8281168814371082, 'eval_f1_weighted': 0.8247668690571658, 'eval_runtime': 0.4862, 'eval_samples_per_second': 493.602, 'eval_steps_per_second': 30.85, 'epoch': 8.0}
📊 Test: {'eval_loss': 0.5246224403381348, 'eval_accuracy': 0.8091286307053942, 'eval_f1_macro': 0.8104762561970332, 'eval_f1_weighted': 0.8092589165596886, 'eval_runtime': 0.5098, 'eval_samples_per_second': 472.747, 'eval_steps_per_second': 31.386, 'epoch': 8.0}

Classification report (test):
               precision    recall  f1-score   support

    negative     0.8571    0.7742    0.8136        31
     neutral     0.7818    0.8037    0.7926       107
    positive     0.8252    0.8252    0.8252       103

    accuracy                         0.8091       241
   macro avg     0.8214    0.8011    0.8105       241
weighted avg     0.8101    0.8091    0.8093       241


💾 Saved best model to: /content/drive/MyDrive/CBDC Project/cbdc-sent

CBDC Type - bert-base-uncased model

In [4]:
# ========================
# 0) Setup & Install
# ========================
# !pip -q install -U transformers datasets evaluate accelerate scikit-learn

import os, json, random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    EarlyStoppingCallback,
    Trainer
)

# Colab Drive
from google.colab import drive
drive.mount('/content/drive')

# ========================
# 1) Paths & Params
# ========================
# Load CSV directly from GitHub (RAW)
DATA_RAW_URL = "https://raw.githubusercontent.com/bilalezafar/CentralBank-BERT/main/5-CBDC-Type/cbdc_type_training.csv"

# Save to: My Drive > CBDC Project > cbdc-type-bert-uncased
OUTPUT_DIR = "/content/drive/MyDrive/CBDC Project/cbdc-type-bert-uncased"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Base model: BERT base uncased
MODEL_NAME = "bert-base-uncased"

SEED       = 42
MAX_LEN    = 192
BATCH_TRAIN= 8
BATCH_EVAL = 16
EPOCHS     = 5
LR         = 2e-5
WARMUP     = 0.1
WEIGHT_DEC = 0.01

# Label schema
label_list = ["Retail CBDC", "Wholesale CBDC", "General/Unspecified"]
label2id = {lbl: i for i, lbl in enumerate(label_list)}
id2label = {i: lbl for lbl, i in label2id.items()}

# Reproducibility
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()

# ========================
# 2) Load & Split
# ========================
df = pd.read_csv(DATA_RAW_URL)

df = df[["sentence", "label"]].dropna().copy()
df["sentence"] = df["sentence"].astype(str)
df["label"] = df["label"].astype(str).map(lambda x: x.strip())

df = df[df["label"].isin(label_list)].reset_index(drop=True)
df["labels"] = df["label"].map(label2id)

print("Counts by label (overall):")
print(df["label"].value_counts())
print("\nTotal rows:", len(df))

train_df, temp_df = train_test_split(
    df, test_size=0.20, stratify=df["labels"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["labels"], random_state=SEED
)
print("\nSplit sizes:", len(train_df), len(val_df), len(test_df))

# ========================
# 3) Hugging Face Datasets
# ========================
ds = DatasetDict({
    "train": Dataset.from_pandas(train_df[["sentence", "labels"]], preserve_index=False),
    "validation": Dataset.from_pandas(val_df[["sentence", "labels"]], preserve_index=False),
    "test": Dataset.from_pandas(test_df[["sentence", "labels"]], preserve_index=False),
})

# ========================
# 4) Tokenizer & Tokenization
# ========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tok_fn(batch):
    return tokenizer(
        batch["sentence"],
        padding=False,
        truncation=True,
        max_length=MAX_LEN,
    )

tokenized = ds.map(tok_fn, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

# ========================
# 5) Class Weights (from TRAIN ONLY)
# ========================
train_counts = train_df["labels"].value_counts().reindex(range(len(label_list)), fill_value=0)
num_classes = len(label_list)
N = len(train_df)

# total/(num_classes * count) with safe guard
weights = {}
for i, cnt in train_counts.items():
    weights[i] = (N / (num_classes * cnt)) if cnt > 0 else 0.0

class_weights_tensor = torch.tensor([weights[i] for i in range(num_classes)], dtype=torch.float)

print("\nClass weights (by id):")
for i in range(num_classes):
    print(f"  {i} ({id2label[i]}): {class_weights_tensor[i]:.3f}")

# ========================
# 6) Model & Config
# ========================
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)

# ========================
# 7) Metrics
# ========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    f1_weighted = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1_macro": f1_macro, "f1_weighted": f1_weighted}

# ========================
# 8) TrainingArguments
# ========================
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    gradient_accumulation_steps=1,
    learning_rate=LR,
    weight_decay=WEIGHT_DEC,
    warmup_ratio=WARMUP,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    fp16=True,
    dataloader_num_workers=2,
    seed=SEED,
)

# ========================
# 9) Weighted Trainer
# ========================
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    # Compatible with recent transformers
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=collator,
    tokenizer=tokenizer,  # safer across versions than processing_class
    compute_metrics=compute_metrics,
    class_weights=class_weights_tensor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=1e-4)],
)

# ========================
# 10) Train
# ========================
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(os.path.join(OUTPUT_DIR, "label_mapping.json"), "w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, indent=2)

# ========================
# 11) Evaluate on Test
# ========================
test_out = trainer.predict(tokenized["test"])
test_preds = test_out.predictions.argmax(axis=-1)

print("\nTest metrics:", test_out.metrics)

report = classification_report(
    test_df["labels"], test_preds, target_names=[id2label[i] for i in range(num_classes)], digits=4
)
cm = confusion_matrix(test_df["labels"], test_preds)

print("\nClassification report:\n", report)
print("\nConfusion matrix (rows=true, cols=pred):\n", cm)

with open(os.path.join(OUTPUT_DIR, "test_classification_report.txt"), "w") as f:
    f.write(report)
np.savetxt(os.path.join(OUTPUT_DIR, "test_confusion_matrix.csv"), cm, fmt="%d", delimiter=",")

print(f"\n✅ Saved model + tokenizer to: {OUTPUT_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Counts by label (overall):
label
General/Unspecified    545
Retail CBDC            543
Wholesale CBDC         329
Name: count, dtype: int64

Total rows: 1417

Split sizes: 1133 142 142


Map:   0%|          | 0/1133 [00:00<?, ? examples/s]

Map:   0%|          | 0/142 [00:00<?, ? examples/s]

Map:   0%|          | 0/142 [00:00<?, ? examples/s]


Class weights (by id):
  0 (Retail CBDC): 0.870
  1 (Wholesale CBDC): 1.436
  2 (General/Unspecified): 0.866


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1422144785.py:183: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.971400,0.460812,0.830986,0.836672,0.830364
2,0.415600,0.347759,0.887324,0.890657,0.887281
3,0.219700,0.378583,0.866197,0.870571,0.865694
4,0.103200,0.456165,0.894366,0.900721,0.894782
5,0.070700,0.458577,0.880282,0.886515,0.880509



Test metrics: {'test_loss': 0.41492944955825806, 'test_accuracy': 0.8661971830985915, 'test_f1_macro': 0.8745845122412144, 'test_f1_weighted': 0.8680942751681069, 'test_runtime': 0.8752, 'test_samples_per_second': 162.249, 'test_steps_per_second': 10.283}

Classification report:
                      precision    recall  f1-score   support

        Retail CBDC     0.9184    0.8182    0.8654        55
     Wholesale CBDC     1.0000    0.8485    0.9180        33
General/Unspecified     0.7692    0.9259    0.8403        54

           accuracy                         0.8662       142
          macro avg     0.8959    0.8642    0.8746       142
       weighted avg     0.8806    0.8662    0.8681       142


Confusion matrix (rows=true, cols=pred):
 [[45  0 10]
 [ 0 28  5]
 [ 4  0 50]]

✅ Saved model + tokenizer to: /content/drive/MyDrive/CBDC Project/cbdc-type-bert-uncased


CBDC Discourse - bert-base-uncased model

In [5]:
# ====================== Colab setup ======================
# !pip -q install -U transformers datasets accelerate evaluate scikit-learn

import os, json, random, numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ====================== Paths & I/O ======================
from google.colab import drive
drive.mount('/content/drive')

# Load CSV directly from GitHub (RAW)
DATA_RAW_URL = "https://raw.githubusercontent.com/bilalezafar/CentralBank-BERT/main/6-CBDC-Discourse/cbdc_classification_training.csv"

# Save to: My Drive > CBDC Project > cbdc-discourse-bert-uncased
BASE_DIR   = "/content/drive/MyDrive/CBDC Project"
OUTPUT_DIR = os.path.join(BASE_DIR, "cbdc-discourse-bert-uncased")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ====================== Load & inspect ======================
df = pd.read_csv(DATA_RAW_URL)

# Handle optional '#' column safely (if present)
if "#" in df.columns:
    df = df.rename(columns={'#': 'row_id'})

# Keep expected columns if present; otherwise fall back safely
keep_cols = [c for c in ["row_id", "url", "sentence", "label"] if c in df.columns]
df = df[keep_cols].copy()

df['sentence'] = df['sentence'].astype(str).str.strip()
df['label']    = df['label'].astype(str).str.strip()
df = df.dropna(subset=['sentence','label']).reset_index(drop=True)

print("=== Sentence Count per Label (full data) ===")
print(df['label'].value_counts())
print("\n=== Percentage Distribution (full data) ===")
print((df['label'].value_counts(normalize=True) * 100).round(2))

# ====================== Label encoding ======================
labels   = sorted(df['label'].unique())
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}
df['label_id'] = df['label'].map(label2id)
print("\nLabel map:", label2id)

# ====================== Stratified splits ======================
train_df, temp_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df['label_id']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label_id']
)

def show_split(name, sdf):
    print(f"\n{name} size: {len(sdf)}")
    print(sdf['label'].value_counts())

show_split("Train (before balance)", train_df)
show_split("Val", val_df)
show_split("Test", test_df)

# ====================== FULL BALANCE the TRAIN split ======================
def full_downsample_balance(train_df, label_col="label", seed=SEED):
    counts = train_df[label_col].value_counts()
    target = counts.min()
    parts = []
    for lab, grp in train_df.groupby(label_col, sort=False):
        if len(grp) > target:
            parts.append(grp.sample(n=target, random_state=seed))
        else:
            parts.append(grp)
    out = pd.concat(parts, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    print("\n[Balance] Train before:\n", counts.to_string())
    print("\n[Balance] Train after:\n", out[label_col].value_counts().to_string())
    return out

train_df_bal = full_downsample_balance(train_df, label_col="label", seed=SEED)

# ====================== HF Datasets ======================
train_df_bal = train_df_bal.assign(label_id=train_df_bal['label'].map(label2id))
val_df       = val_df.assign(label_id=val_df['label'].map(label2id))
test_df      = test_df.assign(label_id=test_df['label'].map(label2id))

# Base model switched to BERT base uncased
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_batch(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        padding=False,
        max_length=MAX_LEN
    )

ds = DatasetDict({
    "train": Dataset.from_pandas(train_df_bal[['sentence','label_id']].rename(columns={'label_id':'labels'}), preserve_index=False),
    "val":   Dataset.from_pandas(val_df[['sentence','label_id']].rename(columns={'label_id':'labels'}), preserve_index=False),
    "test":  Dataset.from_pandas(test_df[['sentence','label_id']].rename(columns={'label_id':'labels'}), preserve_index=False),
})

ds = ds.map(tokenize_batch, batched=True, remove_columns=['sentence'])
collator = DataCollatorWithPadding(tokenizer)

# ====================== Model ======================
from transformers import Trainer
import torch.nn as nn

config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)

class BalancedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels_t = inputs.get("labels")
        outputs = model(**{k:v for k,v in inputs.items() if k != "labels"})
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss()  # train is fully balanced
        loss = loss_fct(logits, labels_t)
        return (loss, outputs) if return_outputs else loss

# ====================== Metrics ======================
def compute_metrics(eval_pred):
    preds, labels_np = eval_pred
    preds = np.argmax(preds, axis=1)
    acc = accuracy_score(labels_np, preds)
    f1_macro = f1_score(labels_np, preds, average="macro")
    f1_weighted = f1_score(labels_np, preds, average="weighted")
    return {"accuracy": acc, "f1_macro": f1_macro, "f1_weighted": f1_weighted}

# ====================== Training arguments ======================
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    log_level="error",
    report_to="none",
    save_total_limit=2,
    dataloader_num_workers=2,
    seed=SEED,
)

callbacks = [EarlyStoppingCallback(early_stopping_patience=2)]

trainer = BalancedTrainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    tokenizer=tokenizer,            # safer across transformers versions
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=callbacks,
)

# ====================== Train ======================
train_out = trainer.train()
print("\nBest checkpoint:", trainer.state.best_model_checkpoint)

# ====================== Evaluate ======================
print("\n=== Validation ===")
val_metrics = trainer.evaluate(eval_dataset=ds["val"])
print(val_metrics)

print("\n=== Test ===")
test_metrics = trainer.evaluate(eval_dataset=ds["test"])
print(test_metrics)

# Detailed test report
preds_logits = trainer.predict(ds["test"]).predictions
test_pred_ids = preds_logits.argmax(axis=1)
y_true = test_df['label_id'].to_numpy()

print("\n--- Classification report (test) ---")
print(classification_report(y_true, test_pred_ids, target_names=labels, digits=4))

# Confusion matrix
cm = confusion_matrix(y_true, test_pred_ids, labels=list(range(len(labels))))
cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix.csv")
cm_df.to_csv(cm_path, index=True)
print(f"\nConfusion matrix saved to: {cm_path}")

# ====================== Save best model & mappings ======================
save_dir = os.path.join(OUTPUT_DIR, "best_model_balanced")
os.makedirs(save_dir, exist_ok=True)
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

with open(os.path.join(save_dir, "label_mapping.json"), "w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "metrics_summary.json"), "w") as f:
    json.dump(
        {"val": val_metrics, "test": test_metrics, "best_checkpoint": trainer.state.best_model_checkpoint},
        f, indent=2
    )

print("\nSaved model to:", save_dir)
print("Artifacts saved under:", OUTPUT_DIR)


Torch: 2.9.0+cu126 | CUDA available: True
GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== Sentence Count per Label (full data) ===
label
Process         2706
Feature         1323
Risk-Benefit    1202
Name: count, dtype: int64

=== Percentage Distribution (full data) ===
label
Process         51.73
Feature         25.29
Risk-Benefit    22.98
Name: proportion, dtype: float64

Label map: {'Feature': 0, 'Process': 1, 'Risk-Benefit': 2}

Train (before balance) size: 4184
label
Process         2164
Feature         1058
Risk-Benefit     962
Name: count, dtype: int64

Val size: 523
label
Process         271
Feature         132
Risk-Benefit    120
Name: count, dtype: int64

Test size: 524
label
Process         271
Feature         133
Risk-Benefit    120
Name: count, dtype: int64

[Balance] Train before:
 label
Process         2164
Feature         1058
Risk-Benefit     962

[Balance] Train after:
 

Map:   0%|          | 0/2886 [00:00<?, ? examples/s]

Map:   0%|          | 0/523 [00:00<?, ? examples/s]

Map:   0%|          | 0/524 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1534518593.py:178: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `BalancedTrainer.__init__`. Use `processing_class` instead.
  trainer = BalancedTrainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,No log,0.526826,0.791587,0.749735,0.785462
2,No log,0.470703,0.824092,0.807060,0.827460
3,0.526700,0.600604,0.816444,0.805947,0.820043
4,0.526700,0.634056,0.845124,0.827366,0.846944
5,0.526700,0.668595,0.850860,0.831666,0.852430
6,0.089200,0.678983,0.843212,0.822723,0.844777



Best checkpoint: /content/drive/MyDrive/CBDC Project/cbdc-discourse-bert-uncased/checkpoint-905

=== Validation ===


{'eval_loss': 0.6685953140258789, 'eval_accuracy': 0.8508604206500956, 'eval_f1_macro': 0.8316662081355183, 'eval_f1_weighted': 0.852429767143464, 'eval_runtime': 1.0558, 'eval_samples_per_second': 495.358, 'eval_steps_per_second': 31.256, 'epoch': 6.0}

=== Test ===
{'eval_loss': 0.9061506986618042, 'eval_accuracy': 0.8015267175572519, 'eval_f1_macro': 0.7810022120456289, 'eval_f1_weighted': 0.8045841367084342, 'eval_runtime': 1.2723, 'eval_samples_per_second': 411.865, 'eval_steps_per_second': 25.938, 'epoch': 6.0}

--- Classification report (test) ---
              precision    recall  f1-score   support

     Feature     0.7246    0.7519    0.7380       133
     Process     0.9106    0.8266    0.8665       271
Risk-Benefit     0.6857    0.8000    0.7385       120

    accuracy                         0.8015       524
   macro avg     0.7736    0.7928    0.7810       524
weighted avg     0.8119    0.8015    0.8046       524


Confusion matrix saved to: /content/drive/MyDrive/CBDC Pr